# Preprocessing Technique: Feature Engineering — Feature Selection & Dimensionality Reduction
### IT2011 — Progress Review I: Data Preprocessing and EDA
**Presented by:** Member 6 — *[Full Name, IT Number]*
**Assigned dataset:** Rotten Tomatoes Movie Review Dataset (Cornell)

This notebook covers my individually-owned preprocessing technique for our group's project,
as required for Progress Review I: technique explanation, justification, implementation, and
an interpreted EDA visualization.


## Shared Setup

This cell is identical across every member's notebook so each person's notebook can run
independently. It loads the assigned dataset and converts it to a pandas DataFrame.


In [ ]:
!pip install -q datasets scikit-learn pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

sns.set_style("whitegrid")

ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")
train_df = ds["train"].to_pandas()
val_df = ds["validation"].to_pandas()
test_df = ds["test"].to_pandas()

print("Train:", train_df.shape, "| Validation:", val_df.shape, "| Test:", test_df.shape)
train_df.head()


## 1. Technique Explanation

**Feature engineering** here covers two related steps: **feature selection** (keeping only
the most informative TF-IDF terms out of thousands of possible words) and **dimensionality
reduction** (compressing that high-dimensional representation down to a small number of
components for visualization and to reduce noise).

## 2. Justification for This Dataset

A TF-IDF matrix over our review text can easily have 5,000–10,000+ columns (one per unique
word/n-gram), most of which are rare and only weakly informative. Selecting the most
statistically relevant features (via a chi-squared test against the sentiment label) reduces
noise and overfitting risk. Separately, reducing the matrix to 2 dimensions via SVD lets us
visually check whether positive and negative reviews are actually separable in feature space
— a useful sanity check before committing to a modeling approach.


## 3. Implementation — Feature Selection

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_tfidf = vectorizer.fit_transform(train_df["text"].str.lower())
y = train_df["label"]

selector = SelectKBest(chi2, k=2000)
X_selected = selector.fit_transform(X_tfidf, y)

selected_features = np.array(vectorizer.get_feature_names_out())[selector.get_support()]
scores = selector.scores_[selector.get_support()]
top_terms = pd.Series(scores, index=selected_features).sort_values(ascending=False).head(15)

print(f"Reduced from {X_tfidf.shape[1]} to {X_selected.shape[1]} features via chi-squared selection.")
print("\nTop 15 most sentiment-informative terms:")
top_terms


## 4. Implementation — Dimensionality Reduction for Visualization

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd.fit_transform(X_selected)

viz_df = pd.DataFrame({"x": X_2d[:, 0], "y": X_2d[:, 1], "label": y.map({0: "Negative", 1: "Positive"})})


## 5. EDA Visualization & Interpretation

In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(data=viz_df.sample(1500, random_state=1), x="x", y="y", hue="label",
                 palette={"Negative": "#C1272D", "Positive": "#2E9E6D"}, alpha=0.5, s=20)
plt.title("2D SVD Projection of Selected TF-IDF Features")
plt.xlabel("SVD Component 1")
plt.ylabel("SVD Component 2")
plt.show()


**Interpretation:** [Fill in after running — describe: is there a visible tendency for
positive (green) and negative (red) points to cluster separately, or do they overlap heavily?
Some overlap is expected and normal — text sentiment is genuinely harder to linearly separate
in just 2 dimensions than a model using the full selected feature set — but any visible
separation trend supports the case that the selected features carry real sentiment signal.]
